# Support Integrity Auditor (SIA)

This notebook runs the full pipeline: load tickets, generate pseudo-labels, train a classifier, evaluate it, and generate grounded Evidence Dossiers.

In [ ]:
import pandas as pd
from sia_core import load_tickets, add_pseudo_labels, train_model, load_model, predict_with_dossiers

DATA_PATH = 'data/customer_support_tickets.csv'
tickets = load_tickets(DATA_PATH)
tickets.head()

## Stage 1: Pseudo-Label Generation

The pipeline combines two independent signals: text urgency and resolution time. The fused inferred severity is compared with the assigned priority to produce `is_mismatch`.

In [ ]:
labeled = add_pseudo_labels(tickets)
labeled[['ticket_id', 'priority', 'inferred_severity', 'severity_delta', 'mismatch_type', 'is_mismatch']].head(10)

## Stage 2: Train Classifier

The classifier uses TF-IDF text features, categorical metadata, resolution time, and class-balanced logistic regression.

In [ ]:
model, metrics, labeled = train_model(tickets, model_dir='models')
metrics

## Stage 3: Evidence Dossiers

Dossiers are generated only from fields in the original ticket. No ungrounded claims are added.

In [ ]:
model = load_model('models/sia_model.pkl')
results = predict_with_dossiers(pd.read_csv('data/adversarial_tickets.csv'), model)
results[['ticket_id', 'priority', 'inferred_severity', 'mismatch_type', 'predicted_mismatch', 'model_confidence']]

In [ ]:
results.loc[results['predicted_mismatch'] == 1, 'dossier'].iloc[0]